# Marine Trash Detection — YOLOv8m-seg
**Life Under Water | VIPS-TC | Krishna Jaiswal**

| | |
|---|---|
| Model | YOLOv8m-seg |
| Dataset | TrashCan 1.0 — 7,212 images, 16 classes |
| Platform | Kaggle GPU (T4 / P100) |
| Target | mAP50 > 75% |

## Cell 1 — Check GPU

In [ ]:
import torch, os

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU     : {gpu.name}')
    print(f'VRAM    : {gpu.total_memory/1e9:.1f} GB')

## Cell 2 — Install Ultralytics

In [ ]:
!pip install ultralytics -q

## Cell 3 — Download Dataset

In [ ]:
import requests, zipfile, glob

def download_file(url, save_name):
    if not os.path.exists(save_name):
        r = requests.get(url, stream=True)
        with open(save_name, 'wb') as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
        print(f'Downloaded: {save_name}')
    else:
        print(f'Already exists: {save_name}')

def unzip(path):
    with zipfile.ZipFile(path) as z:
        z.extractall('./')
    print('Extracted.')

download_file(
    'https://www.dropbox.com/s/ievh0sesad015z0/trash_inst_material.zip?dl=1',
    'trash_inst_material.zip'
)
unzip('trash_inst_material.zip')

train_imgs = glob.glob('trash_inst_material/train/images/*.jpg')
val_imgs   = glob.glob('trash_inst_material/val/images/*.jpg')
print(f'Train: {len(train_imgs)} | Val: {len(val_imgs)}')

## Cell 4 — Create Dataset YAML

In [ ]:
import yaml

cwd = os.getcwd()

config = {
    'path'  : f'{cwd}/trash_inst_material',
    'train' : 'train/images',
    'val'   : 'val/images',
    'nc'    : 16,
    'names' : {
        0: 'rov',               1: 'plant',
        2: 'animal_fish',       3: 'animal_starfish',
        4: 'animal_shells',     5: 'animal_crab',
        6: 'animal_eel',        7: 'animal_etc',
        8: 'trash_etc',         9: 'trash_fabric',
        10: 'trash_fishing_gear', 11: 'trash_metal',
        12: 'trash_paper',      13: 'trash_plastic',
        14: 'trash_rubber',     15: 'trash_wood',
    }
}

with open('trashcan.yaml', 'w') as f:
    yaml.dump(config, f)

class_names = list(config['names'].values())
print('YAML saved.')

## Cell 5 — Class Distribution

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from collections import defaultdict

label_files = glob.glob('trash_inst_material/train/labels/*.txt')
counts = defaultdict(int)

for lf in label_files:
    with open(lf) as f:
        for line in f:
            counts[class_names[int(line.split()[0])]] += 1

names  = list(counts.keys())
values = list(counts.values())
colors = ['#D85A30' if 'trash' in n else '#1D9E75' for n in names]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.barh(names, values, color=colors)
ax.set_xlabel('Instance count')
ax.set_title('Class distribution — TrashCan 1.0 training set')
ax.bar_label(bars, padding=4, fontsize=9)
ax.legend(handles=[
    Patch(color='#D85A30', label='Trash'),
    Patch(color='#1D9E75', label='Marine life')
])
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

## Cell 6 — Train Model

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m-seg.pt')

results = model.train(
    data         = 'trashcan.yaml',
    imgsz        = 640,
    epochs       = 75,
    patience     = 20,
    batch        = 16,
    workers      = 4,
    amp          = True,
    optimizer    = 'SGD',
    lr0          = 0.01,
    lrf          = 0.001,
    momentum     = 0.937,
    weight_decay = 0.0005,
    cos_lr       = True,
    fliplr       = 0.5,
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    mosaic       = 1.0,
    mixup        = 0.1,
    copy_paste   = 0.1,
    degrees      = 10.0,
    translate    = 0.1,
    scale        = 0.5,
    save         = True,
    save_period  = 10,
    name         = 'yolov8m_marine_v2',
    exist_ok     = True,
    plots        = True,
    val          = True,
)

print(f"Best mAP50    : {results.results_dict.get('metrics/mAP50(B)', 'N/A'):.4f}")
print(f"Best mAP50-95 : {results.results_dict.get('metrics/mAP50-95(B)', 'N/A'):.4f}")

## Cell 7 — Plot Training Curves

In [ ]:
import pandas as pd

df = pd.read_csv('runs/segment/yolov8m_marine_v2/results.csv')
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('YOLOv8m-seg — Marine Trash Detection Training', fontsize=14)

plots = [
    ('train/box_loss',       'Box loss',              axes[0, 0]),
    ('train/seg_loss',       'Seg loss',              axes[0, 1]),
    ('train/cls_loss',       'Class loss',            axes[0, 2]),
    ('metrics/mAP50(B)',     'mAP50 — detection',     axes[1, 0]),
    ('metrics/mAP50(M)',     'mAP50 — segmentation',  axes[1, 1]),
    ('metrics/precision(B)', 'Precision',             axes[1, 2]),
]

for col, title, ax in plots:
    if col in df.columns:
        ax.plot(df['epoch'], df[col], color='#065A82', linewidth=2)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## Cell 8 — Evaluate on Validation Set

In [ ]:
model_best = YOLO('runs/segment/yolov8m_marine_v2/weights/best.pt')

val_results = model_best.val(
    data    = 'trashcan.yaml',
    imgsz   = 640,
    batch   = 16,
    verbose = True,
)

print('\nFINAL RESULTS')
print(f'mAP50    detection     : {val_results.box.map50:.4f}')
print(f'mAP50-95 detection     : {val_results.box.map:.4f}')
print(f'mAP50    segmentation  : {val_results.seg.map50:.4f}')
print(f'mAP50-95 segmentation  : {val_results.seg.map:.4f}')

## Cell 9 — Per-Class Results

In [ ]:
print(f"{'Class':<25} {'mAP50':>8}")
print('-' * 35)
for i, name in enumerate(class_names):
    try:
        print(f'{name:<25} {val_results.box.maps[i]:>8.4f}')
    except:
        pass

## Cell 10 — Save Outputs

In [ ]:
import shutil

OUT = '/kaggle/working/marine_trash_v2'
os.makedirs(OUT, exist_ok=True)

files_to_save = [
    'runs/segment/yolov8m_marine_v2/weights/best.pt',
    'runs/segment/yolov8m_marine_v2/results.png',
    'runs/segment/yolov8m_marine_v2/confusion_matrix.png',
    'runs/segment/yolov8m_marine_v2/val_batch0_pred.jpg',
    'training_curves.png',
    'class_distribution.png',
]

for fp in files_to_save:
    if os.path.exists(fp):
        dest = f'{OUT}/{os.path.basename(fp)}'
        shutil.copy(fp, dest)
        print(f'Saved: {os.path.basename(fp)}')

print(f'\nDownload from the Output tab on the right.')

## Cell 11 — Export to ONNX (optional)

In [ ]:
model_best.export(format='onnx', imgsz=640, simplify=True)
print('Exported: best.onnx')